In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df=pd.read_csv("Heart_Disease_Prediction.csv")

In [3]:
df.sample(3)

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
178,67,1,4,125,254,1,0,163,0,0.2,2,2,7,Presence
117,56,0,4,200,288,1,2,133,1,4.0,3,2,7,Presence
239,52,1,2,120,325,0,0,172,0,0.2,1,0,3,Absence


In [4]:
# df.isnull().sum()

In [5]:
# df.duplicated().sum()

In [6]:
# preprocessing

# standardizing data
numeric_cols = ['Age', 'BP', 'Cholesterol', 'Max HR','ST depression', 'Number of vessels fluro']
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

In [7]:
# one hot encoing for cat having values >=3

from sklearn.preprocessing import OneHotEncoder
categorical_cols = ['Chest pain type', 'EKG results', 'Slope of ST', 'Thallium']
encoder = OneHotEncoder(sparse_output=False, drop=None) 

encoded = encoder.fit_transform(df[categorical_cols])

encoded_df = pd.DataFrame(encoded, 
                          columns=encoder.get_feature_names_out(categorical_cols))

df = df.drop(categorical_cols, axis=1)
df = pd.concat([df, encoded_df], axis=1)

In [8]:
df['Heart Disease'] = df['Heart Disease'].map({'Absence': 0, 'Presence': 1})

In [9]:
print(df['Heart Disease'].unique())

[1 0]


In [10]:
# data split
X = df.drop('Heart Disease', axis=1).astype(float).values
y = df['Heart Disease'].astype(float).values.reshape(-1, 1)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)


In [11]:
X_train_np = X_train
y_train_np = y_train.astype(float).reshape(-1, 1)

X_test_np = X_test
y_test_np = y_test.astype(float).reshape(-1, 1)


In [12]:
df.sample(4)

,Age,Sex,BP,Cholesterol,FBS over 120,Max HR,Exercise angina,ST depression,Number of vessels fluro,Heart Disease,...,Chest pain type_4,EKG results_0,EKG results_1,EKG results_2,Slope of ST_1,Slope of ST_2,Slope of ST_3,Thallium_3,Thallium_6,Thallium_7
133,1.052186,1,-0.636310,-0.070929,0,-2.321424,1,1.006048,0.349871,1,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0
91,0.722232,0,-0.075410,1.557280,0,0.835636,0,-0.918565,-0.711535,1,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
123,1.162171,0,1.607289,2.138783,0,0.057183,0,-0.218706,-0.711535,0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
160,-1.807414,1,-0.636310,-0.361681,0,1.397852,1,2.405766,-0.711535,1,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [13]:
df.size

6210

In [14]:
len(df)

270

In [15]:
df.shape

(270, 23)

In [16]:
y_true = df['Heart Disease']

In [17]:
input_dim = X_train.shape[1]
hidden_dim = 16
output_dim = 1


#weight initialization
W1 = np.random.randn(input_dim, hidden_dim) * 0.01
b1 = np.zeros((1, hidden_dim))
W2 = np.random.randn(hidden_dim, output_dim) * 0.01
b2 = np.zeros((1, output_dim))

In [18]:
# actiavtion func

# for output layer
def sigmoid(X):
    X = np.clip(X, -100, 100)
    return 1 / (1 + np.exp(-X))

def tanh(X):
    return np.tanh(X)

def tanh_derivative(X):
    return 1 - np.tanh(X) ** 2

In [19]:
# forward pass
def forward_pass(X):
    Z1 = np.dot(X, W1) + b1
    A1 = tanh(Z1)
    Z2 = np.dot(A1, W2) + b2
    A2 = sigmoid(Z2)
    return Z1, A1, Z2, A2




In [20]:
# loss -> binary_cross entropy
def compute_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-7, 1-1e-7)
    m = y_true.shape[0]
    loss = - (1/m) * np.sum(y_true*np.log(y_pred) + (1-y_true)*np.log(1 - y_pred))
    return loss


In [21]:
# loss = y_true - A2
# print("Predicted output:", y)
# print("Loss:", loss)

In [22]:
def backward_pass(X, y_true, Z1, A1, Z2, A2):
    m = X.shape[0]
    dZ2 = A2 - y_true.reshape(-1,1)
    dW2 = (1/m) * np.dot(A1.T, dZ2)
    db2 = (1/m) * np.sum(dZ2, axis=0, keepdims=True)
    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * tanh_derivative(Z1)
    dW1 = (1/m) * np.dot(X.T, dZ1)
    db1 = (1/m) * np.sum(dZ1, axis=0, keepdims=True)
    return dW1, db1, dW2, db2


In [23]:
# Forward pass
Z1, A1, Z2, A2 = forward_pass(X_train)

# Backward pass
dW1, db1, dW2, db2 = backward_pass(
    X_train, y_train, Z1, A1, Z2, A2
)

# Update weights
learning_rate = 0.1
W1 -= learning_rate * dW1
b1 -= learning_rate * db1
W2 -= learning_rate * dW2
b2 -= learning_rate * db2


In [24]:
# Training loop

epochs = 1000
learning_rate = 0.1 
for i in range(epochs):
    # Use the numpy version of X_train
    Z1, A1, Z2, A2 = forward_pass(X_train_np)
    
    # Compute loss
    loss = compute_loss(y_train_np, A2)
    
    # Backward pass - ensure y_train_np is passed
    dW1, db1, dW2, db2 = backward_pass(X_train_np, y_train_np, Z1, A1, Z2, A2)
    
    # Update weights (This part was already good!)
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

    if i % 100 == 0:
        print(f"Epoch {i}, Loss: {loss:.4f}")

Epoch 0, Loss: 0.6928
Epoch 100, Loss: 0.4142
Epoch 200, Loss: 0.3324
Epoch 300, Loss: 0.3202
Epoch 400, Loss: 0.3140
Epoch 500, Loss: 0.3088
Epoch 600, Loss: 0.3035
Epoch 700, Loss: 0.2978
Epoch 800, Loss: 0.2916
Epoch 900, Loss: 0.2850


In [25]:
_, _, _, y_pred_train = forward_pass(X_train)
y_pred_train_label = (y_pred_train > 0.5).astype(int)

_, _, _, y_pred_test = forward_pass(X_test)
y_pred_test_label = (y_pred_test > 0.5).astype(int)

from sklearn.metrics import accuracy_score
print("Train Accuracy:", accuracy_score(y_train, y_pred_train_label))
print("Test Accuracy:", accuracy_score(y_test, y_pred_test_label))


Train Accuracy: 0.8783068783068783
Test Accuracy: 0.8765432098765432
